# Downloading Hansen Global Forest Change — `lossyear` band

Source: [UMD/hansen/global_forest_change_2024_v1_12](https://developers.google.com/earth-engine/datasets/catalog/UMD_hansen_global_forest_change_2024_v1_12). Native 30m; `lossyear` is the year of forest loss event encoded as **0 = no loss, 1–24 = year 2001–2024**.

Export scale = **100m** to match the other Hansen exports used in this project. The `lossyear` band's default GEE pyramid policy is `mode`, so downsampling preserves the dominant year of loss in the aggregated pixel.

Exports go to Google Drive via Earth Engine. Monitor at the [GEE Task Manager](https://code.earthengine.google.com/tasks).

In [2]:
import ee
import geemap

ee.Authenticate()  # run once if not already authenticated
ee.Initialize()

In [3]:
# Global extent (same bbox used for the other Hansen exports)
world_bbox = ee.Geometry.BBox(-180, -85, 180, 85)

lossyear = ee.Image("UMD/hansen/global_forest_change_2024_v1_12").select("lossyear")

print(lossyear.getInfo()["id"])  # sanity-check

UMD/hansen/global_forest_change_2024_v1_12


In [4]:
# Quick preview — gradient from light yellow (early years) to dark red (recent years)
vis_params = {
    "min": 1,
    "max": 24,
    "palette": ["ffffcc", "ffeda0", "fed976", "feb24c", "fd8d3c", "fc4e2a", "e31a1c", "b10026"],
}

Map = geemap.Map(center=[-7.5, -72.5], zoom=5)
Map.addLayer(lossyear.selfMask().clip(world_bbox), vis_params, "Hansen lossyear (1=2001 ... 24=2024)")
Map.addLayer(world_bbox, {}, "Region")
Map

Map(center=[-7.5, -72.5], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright',…

In [5]:
# Export to Google Drive at 100m (downsampled from native 30m)
task = ee.batch.Export.image.toDrive(
    image=lossyear,
    description="lossyear_100m_30m",
    folder="HANSEN_LOSSYEAR",
    fileNamePrefix="lossyear_100m_30m",
    region=world_bbox,
    scale=100,
    crs="EPSG:4326",
    maxPixels=1e13,
)

task.start()
print("Export task started:", task.id)

Export task started: FPXCARRFXAXVZNJZBW2VO2W6


### NOTE: track the export at the [GEE Task Manager](https://code.earthengine.google.com/tasks)

Drop the output tiles into `maps/raw/Hansen_forest/lossyear/` in the Dropbox project folder, matching the existing layout used by other Hansen-based notebooks.